# SET3-trained NEP: independent test parity

Figure S7 design. Independent800Ry liquid/ice tests only. Absolute predicted and target energies restored with the same fixed training offset; no fitted alignment. Full training RMSEs are reported separately in data/nep_set3/rmse.tsv.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'energy_reference.json').exists(): ROOT=ROOT.parent
DATA=np.load(ROOT/'data/nep_set3/plot_data.npz')
labels={'800Ry': {}}
predictions={'NEP': {}}
for phase in ['liq','ice']:
    e=DATA[phase+'_energy'];f=DATA[phase+'_force']
    labels['800Ry'][phase]={'energy':e[:,1],'force':f[:,3:]}
    predictions['NEP'][phase]={'energy':e[:,0],'force':f[:,:3]}
label_cutoff={'NEP':'800Ry'}
OUT_PNG=ROOT/'figures/nep_set3_test_parity.png'
OUT_PDF=ROOT/'figures/nep_set3_test_parity.pdf'
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'mathtext.cal': 'Times New Roman',
    'mathtext.sf': 'Times New Roman',
    'mathtext.tt': 'Times New Roman',
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})
colors = {'liq': '#12a182', 'ice': '#525288'}
phase_names = {'liq': 'liq', 'ice': 'ice'}
models = ['NEP']
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.6), squeeze=False)

energy_all = np.concatenate([labels['800Ry'][p]['energy'] for p in ['liq','ice']])
elo = np.floor((energy_all.min() - 0.01) * 100) / 100
ehi = np.ceil((energy_all.max() + 0.01) * 100) / 100
flo, fhi = -11.0, 11.0

for row, model in enumerate(models):
    cutoff = label_cutoff[model]
    e_ax = axes[row, 0]
    f_ax = axes[row, 1]
    e_ax.set_title(model + ": energy")
    f_ax.set_title(model + ": forces")
    e_ax.plot([elo, ehi], [elo, ehi], color='black', lw=1.1)
    f_ax.plot([flo, fhi], [flo, fhi], color='black', lw=1.1)

    for phase in ['liq', 'ice']:
        reference = labels[cutoff][phase]
        prediction = predictions[model][phase]
        energy_error = 1000 * (prediction['energy'] - reference['energy'])
        force_error = 1000 * (prediction['force'] - reference['force']).ravel()
        e_rmse = np.sqrt(np.mean(energy_error ** 2))
        f_rmse = np.sqrt(np.mean(force_error ** 2))
        alpha = 1.0 if phase == 'liq' else 0.5
        e_ax.scatter(reference['energy'], prediction['energy'], color=colors[phase], s=18, alpha=alpha, label=f'{phase_names[phase]}: RMSE = {e_rmse:.2f} meV/atom')
        f_ax.scatter(reference['force'].ravel(), prediction['force'].ravel(), color=colors[phase], s=2, alpha=0.15 if phase == 'liq' else 0.08, rasterized=True, label=f'{phase_names[phase]}: RMSE = {f_rmse:.2f} meV/Å')

    e_ax.set_xlim(elo, ehi)
    e_ax.set_ylim(elo, ehi)
    e_ax.ticklabel_format(axis='both', style='plain', useOffset=False)
    f_ax.set_xlim(flo, fhi)
    f_ax.set_ylim(flo, fhi)
    e_ax.set_xlabel(r'$E_{\mathrm{DFT}}$ [eV/atom]')
    f_ax.set_xlabel(r'$F_{\mathrm{DFT}}$ [eV/Å]')
    e_ax.legend(loc='lower right', frameon=True, edgecolor='0.8', handletextpad=0.3, borderpad=0.3)
    force_legend = f_ax.legend(loc='lower right', frameon=True, edgecolor='0.8', handletextpad=0.3, borderpad=0.3)
    # Match the original Figure S6/S7 treatment: the dense force cloud
    # stays transparent, while its legend markers are large and opaque.
    for handle in force_legend.legend_handles:
        handle.set_alpha(1.0)
        handle.set_sizes([28])

    e_inset = e_ax.inset_axes([0.18, 0.73, 0.38, 0.22])
    f_inset = f_ax.inset_axes([0.20, 0.73, 0.38, 0.22])
    for phase in ['liq', 'ice']:
        reference = labels[cutoff][phase]
        prediction = predictions[model][phase]
        for inset, error, bins in [
            (e_inset, 1000 * (prediction['energy'] - reference['energy']), np.linspace(-5, 5, 21)),
            (f_inset, 1000 * (prediction['force'] - reference['force']).ravel(), np.linspace(-500, 500, 51)),
        ]:
            hist, edges = np.histogram(error, bins=bins, density=True)
            inset.plot((edges[:-1] + edges[1:]) / 2, hist, color=colors[phase], lw=1.0)
    e_inset.axvline(0, color='black', lw=0.8, ls='--')
    f_inset.axvline(0, color='black', lw=0.8, ls='--')
    e_inset.set_xlim(-5, 5)
    f_inset.set_xlim(-500, 500)
    e_inset.set_xlabel(r'$\Delta E$ [meV/atom]', fontsize=8, labelpad=1)
    f_inset.set_xlabel(r'$\Delta F$ [meV/Å]', fontsize=8, labelpad=1)
    e_inset.set_ylabel(r'$p(\Delta E)$', fontsize=8, labelpad=1)
    f_inset.set_ylabel(r'$p(\Delta F)$', fontsize=8, labelpad=1)
    e_inset.tick_params(labelsize=7, direction='in')
    f_inset.tick_params(labelsize=7, direction='in')

for row in range(1):
    axes[row, 0].set_ylabel(r'$E_{\mathrm{MLP}}$ [eV/atom]')
    axes[row, 1].set_ylabel(r'$F_{\mathrm{MLP}}$ [eV/Å]')
    for column in range(2):
        axes[row, column].tick_params(direction='in', top=True, right=True)
fig.subplots_adjust(hspace=0.38, wspace=0.34)
fig.savefig(OUT_PNG, dpi=600, bbox_inches='tight')
fig.savefig(OUT_PDF, bbox_inches='tight')

print(OUT_PNG)
print(OUT_PDF)

